In [14]:
import numpy as np

num_landmarks = 3
state_dim = 3 + 2*num_landmarks

state = np.zeros((state_dim, 1))
P = np.eye(state_dim) * 0.1

Q = np.zeros((state_dim, state_dim))
Q[0:3, 0:3] = np.diag([0.05, 0.05, 0.02])
R = np.diag([0.1, 0.05])

true_landmarks = np.array([[5,5], [10,2], [6,8]])
landmark_initialized = [False, False, False]

def motion_model(state, v, w, dt):
    x, y, theta = state[0,0], state[1,0], state[2,0]
    new_state = state.copy()
    new_state[0,0] = x + v*dt*np.cos(theta)
    new_state[1,0] = y + v*dt*np.sin(theta)
    new_state[2,0] = theta + w*dt
    return new_state

def motion_jacobian(state, v, dt):
    theta = state[2,0]
    J = np.eye(state_dim)
    J[0,2] = -v*dt*np.sin(theta)
    J[1,2] = v*dt*np.cos(theta)
    return J

def measurement_jacobian(state, idx):
    x, y, theta = state[0,0], state[1,0], state[2,0]
    lx, ly = state[3+2*idx,0], state[3+2*idx+1,0]
    dx, dy = lx - x, ly - y
    q, r = dx**2 + dy**2, np.sqrt(dx**2 + dy**2)
    
    H = np.zeros((2, state_dim))
    H[0,0:3] = [-dx/r, -dy/r, 0]
    H[1,0:3] = [dy/q, -dx/q, -1]
    H[0,3+2*idx:3+2*idx+2] = [dx/r, dy/r]
    H[1,3+2*idx:3+2*idx+2] = [-dy/q, dx/q]
    return H

dt = 0.1
for step in range(50):
    # Prediction
    v, w = 1.0, 0.1
    state = motion_model(state, v, w, dt)
    P = motion_jacobian(state, v, dt) @ P @ motion_jacobian(state, v, dt).T + Q
    
    # Measurements
    for i in range(num_landmarks):
        # Simulate measurement
        dx = true_landmarks[i,0] - state[0,0]
        dy = true_landmarks[i,1] - state[1,0]
        r_true = np.sqrt(dx**2 + dy**2)
        
        if r_true < 8.0:  # Within sensor range
            bearing_true = np.arctan2(dy, dx) - state[2,0]
            z = np.array([[r_true + np.random.normal(0,0.05)],
                         [bearing_true + np.random.normal(0,0.02)]])
            
            if not landmark_initialized[i]:
                # Initialize landmark
                state[3+2*i,0] = state[0,0] + z[0,0]*np.cos(state[2,0] + z[1,0])
                state[3+2*i+1,0] = state[1,0] + z[0,0]*np.sin(state[2,0] + z[1,0])
                landmark_initialized[i] = True
            else:
                # Update
                dx_l = state[3+2*i,0] - state[0,0]
                dy_l = state[3+2*i+1,0] - state[1,0]
                z_hat = np.array([[np.sqrt(dx_l**2 + dy_l**2)],
                                 [np.arctan2(dy_l, dx_l) - state[2,0]]])
                
                y = z - z_hat
                y[1,0] = np.arctan2(np.sin(y[1,0]), np.cos(y[1,0]))
                
                H = measurement_jacobian(state, i)
                S = H @ P @ H.T + R
                K = P @ H.T @ np.linalg.inv(S)
                
                state = state + K @ y
                P = (np.eye(state_dim) - K @ H) @ P
    
    if step % 10 == 0:
        print(f"\nStep {step}:")
        print(f"  Robot: ({state[0,0]:.2f}, {state[1,0]:.2f}, {state[2,0]:.2f})")
        for j in range(num_landmarks):
            if landmark_initialized[j]:
                print(f"  L{j}: ({state[3+2*j,0]:.2f}, {state[3+2*j+1,0]:.2f})")


Step 0:
  Robot: (0.10, 0.00, 0.01)
  L0: (4.98, 5.00)

Step 10:
  Robot: (1.00, -0.08, 0.09)
  L0: (4.98, 5.01)

Step 20:
  Robot: (1.93, -0.06, 0.18)
  L0: (4.98, 5.01)

Step 30:
  Robot: (2.58, 0.19, 0.25)
  L0: (4.98, 4.99)
  L1: (9.95, 1.96)

Step 40:
  Robot: (3.30, 0.41, 0.35)
  L0: (4.98, 4.99)
  L1: (9.95, 1.96)
